In [ ]:
import pandas as pd 
import sys 
sys.path.append('/home/grace.bcolverd/2024-ESL-BiomassSAR')
from src.fdlbiomass.utils import read_biomass_data

from scipy import ndimage
import pickle
import numpy as np 

: 

In [ ]:

def calculate_covariance_matrix_tutorial(img_stack, outname, kernelsize=10, overwrite=False):
    """
    compute the covariance matrix.
    If the target file already exists and ``overwrite=False`` this function acts as a simple file reader.

    Parameters
    ----------
    img_stack: numpy.ndarray
        the normalized SLC image stack
    outname: str
       the name of the file to be written
    kernelsize: int
        the boxcar smoothing dimension
    overwrite: bool
        overwrite an existing file? Otherwise it is read from file and returned

    Returns
    -------
    numpy.ndarray
        the covariance matrix
    """

    if overwrite or not os.path.isfile(outname):
        rows, cols, nTrack = img_stack.shape

        kernel = np.ones((kernelsize, kernelsize))
        weight = kernel / np.sum(kernel)

        cov_matrix = np.empty((rows, cols, nTrack, nTrack), np.complex64)
        smooth = np.empty((rows, cols, nTrack), np.float32)

        for i in range(0, nTrack):
            mm = np.absolute(img_stack[:, :, i]) ** 2
            smooth[:, :, i] = ndimage.convolve(mm, weight, mode='reflect')

        for i in range(0, nTrack):
            for j in range(0, nTrack):
                # multiply S1 and complex conjugate of S2
                ms = np.multiply(img_stack[:, :, i], np.conjugate(img_stack[:, :, j]))

                ms_real_smooth = ndimage.convolve(ms.real, weight, mode='reflect')
                ms_imag_smooth = complex(0, 1) * ndimage.convolve(ms.imag, weight, mode='reflect')
                ms_smooth = ms_real_smooth + ms_imag_smooth

                cov_matrix[:, :, i, j] = ms_smooth / (smooth[:, :, i] * smooth[:, :, j]) ** 0.5

        # save the variable
        with open(outname, 'wb') as f:
            pickle.dump(cov_matrix, f, 2)
    else:
        with open(outname, 'rb') as f:
            cov_matrix = pickle.load(f)
    return cov_matrix

: 

In [2]:
import os
import numpy as np
from scipy.io import loadmat
from netCDF4 import Dataset

def SLC(parent_path, heading, polarimetric_channel):
    """
    Load TomoSAR data and related parameters.
    
    Parameters:
    parent_path (str): Path to the parent directory
    heading (str): Either 'NW' (290°) or 'SE' (110°)
    polarimetric_channel (str): 'HH', 'HV' or 'VV'
    
    Returns:
    dict: A dictionary containing all loaded data and parameters
    """
    
    # Determine the polarimetric index based on the channel
    pol_ind = {'HH': 1, 'HV': 2, 'VV': 3}[polarimetric_channel]
    
    # Loading scene parameters
    curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'SceneParameters.mat')
    scene_parameters = loadmat(curr_fullpath)
    
    ground_range_axis = scene_parameters['ground_range_axis'].squeeze()
    azimuth_axis = scene_parameters['azimuth_axis'].squeeze()
    Ngr = len(ground_range_axis)
    Naz = len(azimuth_axis)
    
    # Loading phase-to-height factors
    kz = np.zeros((Ngr, Naz, scene_parameters['N'][0][0]), dtype=np.single)
    for n in range(scene_parameters['N'][0][0]):
        filename_kz = scene_parameters['filenames_kz'][n][0]
        if isinstance(filename_kz, np.ndarray):
            filename_kz = str(filename_kz[0])
        curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'kz', filename_kz)
        print(f'Loading {curr_fullpath}...')
        kz[:, :, n] = read_biomass_data(curr_fullpath, [], 2)
        print('done.')
    
    # Loading sensor trajectories
    curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'FocusingCenters.mat')
    focusing_centers = loadmat(curr_fullpath)
    
    # Loading DTMs
    curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'MetaSensingDTM.mat')
    meta_sensing_dtm = loadmat(curr_fullpath)
    curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'TomoDTMsr.mat')
    tomo_dtm_sr = loadmat(curr_fullpath)
    
    # Loading SLC images
    I = np.zeros((Ngr, Naz, scene_parameters['N'][0][0]), dtype=np.complex64)


    for n in range(scene_parameters['N'][0][0]):
        curr_fullpath = os.path.join(parent_path, heading, 'SLC', 'PoliMi', scene_parameters['filenames_slc'][n, pol_ind-1][0])
        print(f'Loading {curr_fullpath}...')
        I[:, :, n] = read_biomass_data(curr_fullpath, [], 1)

        print('done.')

    # Prepare the return dictionary
    return_data = {
        'scene_parameters': scene_parameters,
        'ground_range_axis': ground_range_axis,
        'azimuth_axis': azimuth_axis,
        'kz': kz,
        'focusing_centers': focusing_centers,
        'meta_sensing_dtm': meta_sensing_dtm,
        'tomo_dtm_sr': tomo_dtm_sr,
        'pol_ind': pol_ind, 
        'I_SLC' : I , 
    }
    
    return return_data



In [3]:
parent_path = '/home/grace.bcolverd/esl_dev/tomosar-matlab/P'


# Define heading and polarimetric channel
heading = 'NW'  # Either 'NW' (290°) or 'SE' (110°)
polarimetric_channel = 'HH'

hh_data = SLC(parent_path, 'NW',  'HH')
hv_data = SLC(parent_path,'NW',  'HV')
vv_data = SLC(parent_path, 'NW', 'VV')

Loading /home/grace.bcolverd/esl_dev/tomosar-matlab/P/NW/Util/SLC/kz/kz_20200722071511_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc.dat...
done.
Loading /home/grace.bcolverd/esl_dev/tomosar-matlab/P/NW/Util/SLC/kz/kz_20200722072647_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc.dat...
done.
Loading /home/grace.bcolverd/esl_dev/tomosar-matlab/P/NW/Util/SLC/kz/kz_20200722073834_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc.dat...
done.
Loading /home/grace.bcolverd/esl_dev/tomosar-matlab/P/NW/Util/SLC/kz/kz_20200722075104_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc.dat...
done.
Loading /home/grace.bcolverd/esl_dev/tomosar-matlab/P/NW/Util/SLC/kz/kz_20200722080252_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc.dat...
done.
Loading /home/grace.bcolverd/esl_dev/tomosar-matlab/P/NW/Util/SLC/kz/kz_20200722081445_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc.dat...
done.
Loading /home/grace.bcolverd/esl_dev/tomosar-matlab/P/NW/Util/SLC/kz/kz_20200722082619_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc.dat...
done.
Loading /home/grace.bcolverd/esl_d

In [7]:
hh_data.keys()

dict_keys(['scene_parameters', 'ground_range_axis', 'azimuth_axis', 'kz', 'focusing_centers', 'meta_sensing_dtm', 'tomo_dtm_sr', 'pol_ind', 'I_SLC'])

In [5]:
import numpy as np

def process_topo_phase(I, kz, tomo_dtm_sr, meta_sensing_dtm):
    """
    Process all ground indexes to create ground-steered SAR images.

    Parameters:
    - I: Complex SAR image data (3D array: ground range, azimuth, tomographic)
    - kz: Vertical wavenumbers (3D array: ground range, azimuth, tomographic)
    - tomo_dtm_sr: Dictionary containing 'TomoDTMsr' (2D array: ground range, azimuth)
    - meta_sensing_dtm: Dictionary containing 'MetaSensingDTM' (2D array: ground range, azimuth)

    Returns:
    - Igs: Ground-steered SAR images for all ground indexes (3D array: ground range, azimuth, tomographic)
    """

    # Get dimensions
    Ngr, Naz, Ntomo = I.shape

    # Initialize the output array
    Igs = np.zeros_like(I, dtype=np.complex128)

    for gr_ind in range(Ngr):
        # Ground steering
        dtm_diff = (tomo_dtm_sr['TomoDTMsr'][gr_ind, :] - meta_sensing_dtm['MetaSensingDTM'][gr_ind, :])
        exp_term = np.exp(-1j * kz[gr_ind, :, :] * dtm_diff[:, np.newaxis])
        Igs[gr_ind, :, :] = I[gr_ind, :, :] * exp_term

    if Igs.shape != I.shape:
        raise ValueError('topo phase procesing has changed shape')
    return Igs


In [8]:
sar_data_dict = hh_data
Igs = process_topo_phase(sar_data_dict['I_SLC'], sar_data_dict['kz'], sar_data_dict['tomo_dtm_sr'], sar_data_dict['meta_sensing_dtm'])

In [13]:
C= calculate_covariance_matrix(Igs, outname='test', overwrite=True )
C.shape

MemoryError: Unable to allocate 96.4 GiB for an array with shape (1274, 12953, 28, 28) and data type complex64

In [9]:
Igs.shape

(1274, 12953, 28)

In [29]:
I = hh_data['I_SLC']
kz = hh_data['kz']
tomo_dtm_sr = hh_data['tomo_dtm_sr']
meta_sensing_dtm=hh_data['meta_sensing_dtm']

Naz = I.shape[1]
Ngr = I.shape[0]

print(f"I shape: {I.shape}")
print(f"kz shape: {kz.shape}")
print(f"TomoDTMsr shape: {tomo_dtm_sr['TomoDTMsr'].shape}")
print(f"MetaSensingDTM shape: {meta_sensing_dtm['MetaSensingDTM'].shape}")

# # Local vertical axis (with respect to the local ground level)
# z_axis = np.arange(-60, 61)
# Nz = len(z_axis)

# # Azimuth profile
# S = np.zeros((Nz, Naz))
# # why is ground index defined this way? is it random? 
# gr_ind = round(Ngr/4)
# # normalisation term ? shoud this be 2 x 2 or 1d array - unclear from matlab code - sum over 3rd dim that does not exist? sum over z axis?  single value for each azimuth 
# F = np.sum(np.abs(I[gr_ind, :, :])**2, axis=1)

# # F = np.abs(I[gr_ind, :, :])**2
# print(f"F shape: {F.shape}")
# print(F)
# # Ground steering
# # height diff between two dtms for the given ground index     , across whole azimuth length 
# dtm_diff = (tomo_dtm_sr['TomoDTMsr'][gr_ind, :] - meta_sensing_dtm['MetaSensingDTM'][gr_ind, :])
# print('dtm ' , dtm_diff.shape)
# # this kz[gr_ind, :, :] selects the corresponding vertical wavenumbers for the ground range index for all azimuth, then mult the  vertical wavenumbers by the height difference for each azimuth, creating a phase correction term.
# exp_term = np.exp(-1j * kz[gr_ind, :, :] * dtm_diff[:, np.newaxis])
# print('exp term ' , exp_term.shape)
# # this selects a slice of the complex SAR image data I for gr_ind, all elements along the azimuth and tomographic dimensions for this ground range. mult by exp term (exp has all azimuth and all z for one gr ind) 
# Igs = I[gr_ind, :, :] * exp_term


I shape: (1274, 12953, 28)
kz shape: (1274, 12953, 28)
TomoDTMsr shape: (1274, 12953)
MetaSensingDTM shape: (1274, 12953)
F shape: (12953,)
[0.04167349 0.09284282 0.12392019 ... 4.3905816  3.4761431  2.569966  ]
dtm  (12953,)
exp term  (12953, 28)
S shape: (121, 12953)


In [11]:
# from src.fdlbiomass.tomograph_reconstruction.utils import * 

# I = hh_data['I_SLC']
# kz = hh_data['kz']
# tomo_dtm_sr = hh_data['tomo_dtm_sr']
# meta_sensing_dtm=hh_data['meta_sensing_dtm']

# # Perform tomographic phase removal for the entire image
# S_full = tomographic_phase_removal(I, kz, tomo_dtm_sr, meta_sensing_dtm)

# # Plot a tomographic slice for a specific ground range index (e.g., gr_ind = 100)
# plot_tomographic_slice(S_full, meta_sensing_dtm, np.linspace(-60, 60, 121), gr_ind=100)

# # To visualize multiple slices or create a video, you can loop through different gr_ind values
# for gr_ind in range(0, S_full.shape[1], 10):  # Plot every 10th slice
#     plot_tomographic_slice(S_full, meta_sensing_dtm, np.linspace(-60, 60, 121), gr_ind)

In [ ]:
# dump ti pickle 
S_full

In [17]:
hh_data['I_SLC'].shape, hh_data['kz'].shape,  hh_data['tomo_dtm_sr']['TomoDTMsr'].shape,  hh_data['meta_sensing_dtm']['MetaSensingDTM'].shape

((1274, 12953, 28), (1274, 12953, 28), (1274, 12953), (1274, 12953))

In [ ]:
# import pickle 
# import scipy.ndimage 
# def topo_phase_removal(img_stack, dem_stack, outname, overwrite=False):
#     """
#     Removal of Topographical Phase.
#     If the target file already exists and ``overwrite=False`` this function acts as a simple file reader.

#     Parameters
#     ----------
#     img_stack: numpy.ndarray
#         the SLC image stack
#     dem_stack: numpy.ndarray
#         the image stack containing flat earth and topographic phase
#     outname: str
#         the name of the file to be written
#     overwrite: bool
#         overwrite an existing file? Otherwise it is read from file and returned

#     Returns
#     -------
#     numpy.ndarray
#         the normalized SLC stack
#     """
#     if overwrite or not os.path.isfile(outname):

#         normalized_stack = img_stack * np.exp(complex(0, 1) * dem_stack, dtype=np.complex64)

#         # save the variable
#         with open(outname, 'wb') as f:
#             pickle.dump(normalized_stack, f, 2)
#     else:
#         with open(outname, 'rb') as f:
#             normalized_stack = pickle.load(f)
#     return normalized_stack


In [19]:
kz = hh_data['kz']
kz.shape

(1274, 12953, 28)

In [ ]:
h_mini = hh_data['I_SLC'][0:100, 0:100, : ]
hv_mini = hv_data['I_SLC'][0:100, 0:100, : ]
vv_mini = vv_data['I_SLC'][0:100, 0:100, : ]


trio_stack = np.concatenate((h_mini, hv_mini, vv_mini, vv_mini), axis=2)
trio_stack.shape

(100, 100, 112)

In [27]:
one_pix = vv_mini[0,0,:]
op_h = np.conjugate(one_pix)
one_pix
np.isreal(op_h *  one_pix)

array([-0.04610366+0.01527326j, -0.1440648 -0.07207184j,
       -0.02714717+0.00832345j, -0.08717734+0.03912172j,
       -0.09117251+0.11797406j, -0.31155193+0.05740336j,
       -0.06983536+0.14278662j, -0.18703082-0.01647791j,
       -0.01516484-0.06744187j, -0.10802689-0.03415048j,
        0.00441156+0.19958705j, -0.03085605+0.14300685j,
       -0.04298878+0.00071338j,  0.0294716 -0.09011386j,
       -0.07216115+0.00189041j,  0.03164009-0.12666805j,
        0.06936675-0.12796445j, -0.14468618-0.06973349j,
       -0.03146605+0.06255431j, -0.04975704-0.01350705j,
       -0.08701283+0.05257399j,  0.03346092-0.01094627j,
       -0.13948146+0.03543606j, -0.0701777 +0.02431529j,
       -0.01454737-0.01421136j,  0.03586283-0.03145851j,
        0.01742055-0.02145692j, -0.16641805+0.20319453j], dtype=complex64)

In [36]:
result = op_h * one_pix
print(np.abs(np.imag(result)))

result = op_h * one_pix
print(np.allclose(np.imag(result), 0))

result = op_h * one_pix
rounded_result = np.round(result, decimals=10)
print(np.isreal(rounded_result))

result = op_h * one_pix
real_result = np.real_if_close(result)
print(np.isreal(real_result))

[2.54124499e-11 5.15977261e-11 6.83436641e-12 9.61872804e-11
 2.01736294e-10 7.40054240e-10 4.11837675e-10 6.97991664e-11
 4.79006418e-11 6.74965639e-11 1.72866443e-11 1.02422099e-10
 2.08676175e-13 5.34393779e-11 4.30462523e-12 1.55018054e-10
 2.41827003e-10 4.58483695e-10 9.16612342e-11 1.09845258e-11
 6.52109478e-11 3.37507799e-12 8.10962408e-12 5.10904097e-11
 1.78984778e-12 1.04979775e-11 6.01727002e-13 1.49535406e-09]
True
[ True False  True False False False False False  True False  True False
  True False  True False False False False  True False  True  True False
  True  True  True False]
[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True]


In [ ]:
hh_data['I_SLC']

C= calculate_covariance_matrix(hh_data['I_SLC'][0:100, 0:100,: ], outname='test', overwrite=True )
C.shape

In [11]:
C= calculate_covariance_matrix(vv_mini, outname='test', overwrite=True )
C.shape

(100, 100, 28, 28)

In [42]:
import numpy as np

def process_covariance_matrices(cov_matrices):
    x_dim, y_dim, n_slcs, _ = cov_matrices.shape
    result = np.zeros((x_dim, y_dim, 84), dtype=np.float32)

    for x in range(x_dim):
        for y in range(y_dim):
            # Extract the covariance matrix for this x,y position
            cov_matrix = cov_matrices[x, y]
            
            # Extract the first row
            first_row = cov_matrix[0]
            
            # Extract the diagonal
            diag = np.diagonal(cov_matrix)
            
            # Split real and imaginary parts of the first row
            first_row_real = np.real(first_row)
            first_row_imag = np.imag(first_row)
            
            # Stack all parts
            result[x, y] = np.concatenate((first_row_real, first_row_imag, diag))
    
    return result



In [44]:
stsacks = process_covariance_matrices(C)

/var/tmp/ipykernel_1733957/3881615108.py:23: ComplexWarning: Casting complex values to real discards the imaginary part
  result[x, y] = np.concatenate((first_row_real, first_row_imag, diag))


In [46]:
stsacks.shape

(100, 100, 84)

In [13]:
input_img = vv_mini
C= calculate_covariance_matrix(input_img, outname='test', overwrite=True )
n_track = input_img.shape[2]
C.shape
row = C[:,:,0,: ]
diag = C[:, : ]

(100, 100, 28, 28)

In [40]:
def test_hermitian(cov_matrix):
    return np.allclose(cov_matrix, np.conjugate(np.swapaxes(cov_matrix, 2, 3)))

def test_positive_semidefinite(cov_matrix):
    for i in range(cov_matrix.shape[0]):
        for j in range(cov_matrix.shape[1]):
            eigvals = np.linalg.eigvals(cov_matrix[i, j])
            if not np.all(eigvals >= -1e-10):  # Allow for small numerical errors
                return False
    return True

# def test_diagonal_real_positive(cov_matrix):
#     diag = np.diagonal(cov_matrix, axis1=2, axis2=3)
#     return np.all(np.isreal(diag)) and np.all(diag > 0)

    

def test_diagonal_real_positive(cov_matrix):
    """  test if diagnoal real if clost 
    """
    diag = np.diagonal(cov_matrix, axis1=2, axis2=3)
    real_diag = np.real_if_close(diag)
    return np.all(np.isreal(real_diag)) and np.all(real_diag > 0)

# def test_correlation_coefficients(cov_matrix):
#     diag = np.diagonal(cov_matrix, axis1=2, axis2=3)
#     std_dev = np.sqrt(diag)
#     corr_matrix = cov_matrix / np.outer(std_dev, std_dev).transpose(0, 2, 1, 3)
#     return np.all(np.abs(corr_matrix) <= 1 + 1e-10)  # Allow for small numerical errors

import numpy as np

def test_correlation_coefficients(cov_matrix):
    n = cov_matrix.shape[0]
    m = cov_matrix.shape[2]
    
    # Compute standard deviations
    std_dev = np.sqrt(np.diagonal(cov_matrix, axis1=2, axis2=3))
    
    for i in range(n):
        for j in range(m):
            for k in range(m):
                if j == k:
                    continue  # Diagonal elements are always 1, no need to check
                corr = cov_matrix[i, :, j, k] / (std_dev[i, :, j] * std_dev[i, :, k])
                if np.any(np.abs(corr) > 1 + 1e-10):
                    return False
    
    return True

def test_trace_property(cov_matrix):
    trace = np.trace(cov_matrix, axis1=2, axis2=3)
    diag_sum = np.sum(np.diagonal(cov_matrix, axis1=2, axis2=3), axis=2)
    return np.allclose(trace, diag_sum)




In [41]:
print(test_trace_property(C))
print(test_diagonal_real_positive(C))
print(test_hermitian(C))
print(test_correlation_coefficients(C))

True
True
True
True


In [21]:
cov_matrix=C
diag = np.diagonal(cov_matrix, axis1=2, axis2=3)
diag
np.all(np.isreal(diag)) 

np.False_

In [ ]:
from scipy.signal import convolve2d
import numpy as np

def integrate_polarimetric_multilook(HH, HV, VV, window_size=(5, 5)):
    X, Y, N = HH.shape
    
    # Ensure all inputs have the same shape
    assert HH.shape == HV.shape == VV.shape, "All input arrays must have the same shape"
    
    # Initialize the output covariance matrix
    cov_matrix = np.zeros((X, Y, 3*N, 3*N), dtype=np.complex64)
    
    # Create a window for spatial averaging
    window = np.ones(window_size) / (window_size[0] * window_size[1])
    
    # Compute the covariance matrix elements
    for i in range(N):
        for j in range(N):
            # HH-HH* element
            cov_matrix[:,:,i,j] = convolve2d(HH[:,:,i] * np.conj(HH[:,:,j]), window, mode='same', boundary='symm')
            
            # HH-HV* element
            cov_matrix[:,:,i,j+N] = convolve2d(HH[:,:,i] * np.conj(HV[:,:,j]), window, mode='same', boundary='symm')
            
            # HH-VV* element
            cov_matrix[:,:,i,j+2*N] = convolve2d(HH[:,:,i] * np.conj(VV[:,:,j]), window, mode='same', boundary='symm')
            
            # HV-HH* element
            cov_matrix[:,:,i+N,j] = np.conj(cov_matrix[:,:,i,j+N])
            
            # HV-HV* element
            cov_matrix[:,:,i+N,j+N] = convolve2d(HV[:,:,i] * np.conj(HV[:,:,j]), window, mode='same', boundary='symm')
            
            # HV-VV* element
            cov_matrix[:,:,i+N,j+2*N] = convolve2d(HV[:,:,i] * np.conj(VV[:,:,j]), window, mode='same', boundary='symm')
            
            # VV-HH* element
            cov_matrix[:,:,i+2*N,j] = np.conj(cov_matrix[:,:,i,j+2*N])
            
            # VV-HV* element
            cov_matrix[:,:,i+2*N,j+N] = np.conj(cov_matrix[:,:,i+N,j+2*N])
            
            # VV-VV* element
            cov_matrix[:,:,i+2*N,j+2*N] = convolve2d(VV[:,:,i] * np.conj(VV[:,:,j]), window, mode='same', boundary='symm')
    
    return cov_matrix


# Usage
HH = h_mini
HV = hv_mini
VV = vv_mini

integrated_cov = integrate_polarimetric_multilook(HH, HV, VV)